# 📸 Clicksy — KNN Resale Price Predictor
### Final Year Project | Photography Gear Marketplace (India)

**Pipeline:**
1. Install dependencies
2. Build dataset (Camera Body, Lens, Drone, Lighting, Audio, Stabilizer, Accessory)
3. Feature engineering
4. Train KNN with cross-validated K selection
5. Evaluate (MAE, MAPE, R²)
6. Save model as `.pkl`
7. Upload to Hugging Face

## Step 1 — Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy joblib -q

## Step 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split, KFold
from sklearn.metrics import mean_absolute_error, r2_score

print("All libraries imported successfully ✅")

## Step 3 — Build Dataset
All 7 categories from Clicksy marketplace dropdown.
Prices based on Amazon India, Flipkart, and Indian photography community resale data.
Depreciation curve modelled from OLX and MPB India listings.

In [ ]:
# ── Reference items: (brand, category, launch_year, new_price_inr) ──────────
ITEMS = [
    # CAMERA BODY
    ("Sony",     "Camera Body", 2021, 234990),  # A7 IV
    ("Sony",     "Camera Body", 2019, 159990),  # A7 III
    ("Sony",     "Camera Body", 2022, 84990),   # ZV-E10 II
    ("Sony",     "Camera Body", 2020, 299990),  # A7S III
    ("Sony",     "Camera Body", 2018, 74990),   # A6400
    ("Sony",     "Camera Body", 2017, 64990),   # A6300
    ("Sony",     "Camera Body", 2023, 319990),  # A9 III
    ("Sony",     "Camera Body", 2016, 54990),   # A6000
    ("Canon",    "Camera Body", 2022, 164990),  # R6 Mark II
    ("Canon",    "Camera Body", 2021, 334990),  # R5
    ("Canon",    "Camera Body", 2020, 79990),   # M50 Mark II
    ("Canon",    "Camera Body", 2019, 54990),   # 90D
    ("Canon",    "Camera Body", 2022, 59990),   # R50
    ("Canon",    "Camera Body", 2023, 239990),  # R8
    ("Canon",    "Camera Body", 2017, 44990),   # 800D
    ("Canon",    "Camera Body", 2016, 34990),   # 750D
    ("Nikon",    "Camera Body", 2021, 549990),  # Z9
    ("Nikon",    "Camera Body", 2022, 349990),  # Z8
    ("Nikon",    "Camera Body", 2020, 134990),  # Z6 II
    ("Nikon",    "Camera Body", 2019, 94990),   # Z50
    ("Nikon",    "Camera Body", 2021, 64990),   # Zfc
    ("Nikon",    "Camera Body", 2018, 84990),   # D7500
    ("Nikon",    "Camera Body", 2017, 74990),   # D500
    ("Nikon",    "Camera Body", 2016, 54990),   # D5600
    ("Fujifilm", "Camera Body", 2022, 154990),  # X-H2
    ("Fujifilm", "Camera Body", 2022, 249990),  # X-H2S
    ("Fujifilm", "Camera Body", 2021, 99990),   # X-T4
    ("Fujifilm", "Camera Body", 2023, 79990),   # X-S20
    ("Fujifilm", "Camera Body", 2020, 59990),   # X-T30 II
    ("Fujifilm", "Camera Body", 2019, 44990),   # X-T200
    ("Fujifilm", "Camera Body", 2017, 39990),   # X-T20
    ("Leica",    "Camera Body", 2022, 799990),  # Q3
    ("Leica",    "Camera Body", 2020, 649990),  # M10-R
    ("Leica",    "Camera Body", 2023, 999990),  # SL3
    ("Leica",    "Camera Body", 2019, 549990),  # CL
    ("GoPro",    "Camera Body", 2023, 44990),   # Hero 12
    ("GoPro",    "Camera Body", 2022, 34990),   # Hero 11
    ("GoPro",    "Camera Body", 2021, 29990),   # Hero 10
    ("GoPro",    "Camera Body", 2019, 24990),   # Hero 8
    ("GoPro",    "Camera Body", 2018, 19990),   # Hero 7
    # LENS
    ("Sony",     "Lens", 2021, 134990),
    ("Sony",     "Lens", 2019, 94990),
    ("Sony",     "Lens", 2017, 64990),
    ("Sony",     "Lens", 2022, 74990),
    ("Sony",     "Lens", 2018, 84990),
    ("Sony",     "Lens", 2020, 34990),
    ("Sony",     "Lens", 2019, 44990),
    ("Sony",     "Lens", 2017, 24990),
    ("Canon",    "Lens", 2020, 109990),
    ("Canon",    "Lens", 2019, 79990),
    ("Canon",    "Lens", 2021, 149990),
    ("Canon",    "Lens", 2022, 39990),
    ("Canon",    "Lens", 2020, 54990),
    ("Canon",    "Lens", 2018, 44990),
    ("Canon",    "Lens", 2017, 34990),
    ("Nikon",    "Lens", 2021, 84990),
    ("Nikon",    "Lens", 2020, 59990),
    ("Nikon",    "Lens", 2022, 99990),
    ("Nikon",    "Lens", 2019, 44990),
    ("Nikon",    "Lens", 2018, 54990),
    ("Fujifilm", "Lens", 2021, 74990),
    ("Fujifilm", "Lens", 2019, 54990),
    ("Fujifilm", "Lens", 2022, 44990),
    ("Fujifilm", "Lens", 2018, 34990),
    ("Fujifilm", "Lens", 2017, 29990),
    ("Leica",    "Lens", 2020, 299990),
    ("Leica",    "Lens", 2019, 449990),
    ("Leica",    "Lens", 2021, 199990),
    ("Leica",    "Lens", 2018, 349990),
    # DRONE
    ("DJI", "Drone", 2023, 159990),
    ("DJI", "Drone", 2022, 109990),
    ("DJI", "Drone", 2023, 54990),
    ("DJI", "Drone", 2020, 94990),
    ("DJI", "Drone", 2022, 49990),
    ("DJI", "Drone", 2019, 129990),
    ("DJI", "Drone", 2021, 139990),
    ("DJI", "Drone", 2023, 74990),
    ("DJI", "Drone", 2018, 84990),
    ("DJI", "Drone", 2017, 99990),
    ("DJI", "Drone", 2020, 44990),
    ("GoPro", "Drone", 2022, 79990),
    # LIGHTING
    ("Profoto", "Lighting", 2021, 99990),
    ("Profoto", "Lighting", 2019, 149990),
    ("Profoto", "Lighting", 2022, 74990),
    ("Profoto", "Lighting", 2020, 54990),
    ("Profoto", "Lighting", 2018, 199990),
    ("Godox",   "Lighting", 2022, 24990),
    ("Godox",   "Lighting", 2021, 14990),
    ("Godox",   "Lighting", 2020, 8990),
    ("Godox",   "Lighting", 2022, 34990),
    ("Godox",   "Lighting", 2019, 44990),
    ("Godox",   "Lighting", 2021, 19990),
    ("Godox",   "Lighting", 2020, 12990),
    ("Godox",   "Lighting", 2023, 29990),
    ("Godox",   "Lighting", 2018, 6990),
    # AUDIO
    ("Rode", "Audio", 2022, 34990),
    ("Rode", "Audio", 2023, 44990),
    ("Rode", "Audio", 2021, 24990),
    ("Rode", "Audio", 2020, 14990),
    ("Rode", "Audio", 2019, 9990),
    ("Rode", "Audio", 2022, 19990),
    ("Rode", "Audio", 2020, 29990),
    ("Rode", "Audio", 2018, 12990),
    ("Rode", "Audio", 2017, 7990),
    ("Rode", "Audio", 2021, 8990),
    # STABILIZER
    ("DJI",   "Stabilizer", 2022, 39990),
    ("DJI",   "Stabilizer", 2023, 29990),
    ("DJI",   "Stabilizer", 2021, 24990),
    ("DJI",   "Stabilizer", 2023, 19990),
    ("DJI",   "Stabilizer", 2020, 34990),
    ("DJI",   "Stabilizer", 2019, 29990),
    ("DJI",   "Stabilizer", 2018, 24990),
    ("Godox", "Stabilizer", 2021, 14990),
    ("DJI",   "Stabilizer", 2022, 44990),
    ("DJI",   "Stabilizer", 2021, 17990),
    # ACCESSORY
    ("Sony",     "Accessory", 2022, 9990),
    ("Sony",     "Accessory", 2021, 7990),
    ("Canon",    "Accessory", 2022, 7990),
    ("Canon",    "Accessory", 2021, 5990),
    ("Nikon",    "Accessory", 2021, 6990),
    ("Nikon",    "Accessory", 2020, 4990),
    ("GoPro",    "Accessory", 2022, 4990),
    ("GoPro",    "Accessory", 2021, 5990),
    ("DJI",      "Accessory", 2022, 8990),
    ("DJI",      "Accessory", 2021, 6990),
    ("Rode",     "Accessory", 2021, 3990),
    ("Godox",    "Accessory", 2021, 4990),
    ("Profoto",  "Accessory", 2020, 12990),
    ("Fujifilm", "Accessory", 2021, 5990),
]

CONDITIONS = [
    ("New (Open Box)", 6, 0.90),
    ("Like New",       5, 0.80),
    ("Excellent",      4, 0.68),
    ("Good",           3, 0.55),
    ("Fair",           2, 0.38),
    ("For Parts",      1, 0.18),
]

def depreciation(age):
    table = {0:0.88, 1:0.74, 2:0.63, 3:0.54, 4:0.46, 5:0.39, 6:0.33, 7:0.28}
    return table.get(age, max(0.22, 0.28 - (age - 7) * 0.03))

rng = np.random.default_rng(42)
records = []

for (brand, category, launch_year, new_price) in ITEMS:
    for sale_year in [2021, 2022, 2023, 2024, 2025]:
        age = sale_year - launch_year
        if age < 0:
            continue
        dep = depreciation(age)
        for (condition, cond_score, cond_mult) in CONDITIONS:
            if condition in ("New (Open Box)", "Like New") and age > 1:
                continue
            if condition == "For Parts" and age < 2:
                continue
            base  = new_price * dep * cond_mult
            noise = rng.normal(1.0, 0.06)
            price = max(500, round(base * noise / 500) * 500)
            records.append({
                "brand":             brand,
                "category":          category,
                "launch_year":       launch_year,
                "sale_year":         sale_year,
                "age_at_sale":       age,
                "condition":         condition,
                "condition_score":   cond_score,
                "new_price_inr":     new_price,
                "resale_price_inr":  int(price),
            })

df = pd.DataFrame(records)
df.to_csv("clicksy_dataset.csv", index=False)
print(f"✅ Dataset built: {len(df)} records across {df['category'].nunique()} categories")
print()
print(df.groupby("category")[["resale_price_inr"]].agg(["count","mean","min","max"]).round(0))

## Step 4 — Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price distribution per category
df.boxplot(column="resale_price_inr", by="category", ax=axes[0], rot=30)
axes[0].set_title("Resale Price Distribution by Category")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Price (₹)")

# Condition vs Price
df.groupby("condition")["resale_price_inr"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="steelblue"
)
axes[1].set_title("Average Resale Price by Condition")
axes[1].set_xlabel("Avg Price (₹)")

plt.suptitle("Clicksy Dataset — EDA", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_plot.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved as eda_plot.png")

## Step 5 — Feature Engineering
KNN requires all features to be **numeric**. We encode:
- `condition` → ordinal score 1–6 (already in dataset)
- `category`  → integer label (label encoding)
- `brand`     → brand tier score (premium positioning)
- `new_price` → log transform (compresses ₹5K–₹10L range)
- `age_at_sale` → raw years (direct depreciation proxy)

In [ ]:
BRAND_TIER = {
    "Leica":    2.5,
    "Profoto":  2.2,
    "Sony":     1.8,
    "Canon":    1.7,
    "Nikon":    1.7,
    "Fujifilm": 1.5,
    "DJI":      1.4,
    "Rode":     1.3,
    "GoPro":    1.1,
    "Godox":    0.9,
    "Other":    0.85,
}

CATEGORY_LIST = [
    "Camera Body", "Lens", "Drone",
    "Lighting", "Audio", "Stabilizer", "Accessory"
]

df["brand_tier"]       = df["brand"].map(BRAND_TIER).fillna(0.85)
df["log_new_price"]    = np.log1p(df["new_price_inr"])
df["category_encoded"] = df["category"].map(
    {cat: i for i, cat in enumerate(CATEGORY_LIST)}
).fillna(len(CATEGORY_LIST))

FEATURES = ["condition_score", "age_at_sale", "log_new_price",
            "category_encoded", "brand_tier"]
TARGET   = "resale_price_inr"

X = df[FEATURES].values
y = df[TARGET].values

print("Features:", FEATURES)
print(f"X shape: {X.shape}")
print(f"y range: ₹{y.min():,} — ₹{y.max():,}")
print()
print("Feature preview:")
print(df[FEATURES].head(8).to_string(index=False))

## Step 6 — Train / Test Split (80 / 20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

## Step 7 — Cross-Validated K Selection
We test **K ∈ {3, 5, 7, 9, 11, 13, 15}** using 5-fold cross-validation on the training set.
The K with the lowest Mean Absolute Error (MAE) is selected.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

k_values = [3, 5, 7, 9, 11, 13, 15]
cv_results = []

print(f"{'K':>4} | {'CV MAE (₹)':>12} | {'CV Std (₹)':>11}")
print("-" * 34)

for k in k_values:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn",    KNeighborsRegressor(n_neighbors=k,
                                        metric="euclidean",
                                        weights="distance")),
    ])
    scores = cross_val_score(pipe, X_train, y_train,
                             cv=kf, scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    std = scores.std()
    cv_results.append((k, mae, std))
    print(f"{k:>4} | {mae:>12,.0f} | {std:>11,.0f}")

best_k, best_mae, _ = min(cv_results, key=lambda x: x[1])
print(f"\n✅ Best K = {best_k}  (CV MAE = ₹{best_mae:,.0f})")

# Plot CV results
ks    = [r[0] for r in cv_results]
maes  = [r[1] for r in cv_results]
stds  = [r[2] for r in cv_results]

plt.figure(figsize=(8, 4))
plt.errorbar(ks, maes, yerr=stds, marker='o', color='royalblue',
             capsize=5, linewidth=2, label='CV MAE ± std')
plt.axvline(best_k, color='crimson', linestyle='--', label=f'Best K={best_k}')
plt.xlabel("K (Number of Neighbours)")
plt.ylabel("MAE (₹)")
plt.title("KNN — Cross-Validated K Selection")
plt.legend()
plt.xticks(ks)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("k_selection.png", dpi=120, bbox_inches="tight")
plt.show()

## Step 8 — Train Final KNN Model

In [ ]:
final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsRegressor(n_neighbors=best_k,
                                    metric="euclidean",
                                    weights="distance")),
])

final_pipeline.fit(X_train, y_train)
print(f"✅ Model trained with K={best_k}")
print(f"   Scaler: StandardScaler (zero mean, unit variance per feature)")
print(f"   Distance metric: Euclidean")
print(f"   Prediction: Inverse-distance weighted average of {best_k} neighbours")

## Step 9 — Model Evaluation on Test Set

In [ ]:
y_pred = final_pipeline.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("=" * 40)
print("  CLICKSY KNN — TEST SET METRICS")
print("=" * 40)
print(f"  MAE   = ₹{mae:,.0f}")
print(f"  MAPE  = {mape:.1f}%")
print(f"  R²    = {r2:.4f}")
print("=" * 40)

# Actual vs Predicted scatter plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=20)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel("Actual Price (₹)")
plt.ylabel("Predicted Price (₹)")
plt.title(f"KNN Predictions vs Actual  |  R²={r2:.3f}  MAE=₹{mae:,.0f}")
plt.legend()
plt.tight_layout()
plt.savefig("predictions_plot.png", dpi=120, bbox_inches="tight")
plt.show()

# Per-category error breakdown
df_test = pd.DataFrame(X_test, columns=FEATURES)
df_test["actual"]    = y_test
df_test["predicted"] = y_pred
df_test["abs_error"] = np.abs(y_test - y_pred)
df_test["category"]  = df_test["category_encoded"].map(
    {i: cat for i, cat in enumerate(CATEGORY_LIST)}
)
print()
print("Per-Category MAE:")
print(df_test.groupby("category")["abs_error"].mean().round(0).sort_values().to_string())

## Step 10 — Save Model as `.pkl`

In [ ]:
# Save the full sklearn pipeline (scaler + KNN together)
joblib.dump(final_pipeline, "clicksy_knn_model.pkl")

# Save metadata the API needs
metadata = {
    "features":       FEATURES,
    "best_k":         int(best_k),
    "brand_tier":     BRAND_TIER,
    "category_list":  CATEGORY_LIST,
    "mae":            int(round(mae)),
    "r2":             round(r2, 4),
    "mape":           round(mape, 1),
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ clicksy_knn_model.pkl  — sklearn Pipeline (StandardScaler + KNN)")
print("✅ model_metadata.json   — feature names, brand tiers, metrics")
print("✅ clicksy_dataset.csv   — full training dataset")
print()
print("Download these 3 files for Hugging Face deployment ⬇️")

## Step 11 — Download Files from Colab

In [ ]:
from google.colab import files

files.download("clicksy_knn_model.pkl")
files.download("model_metadata.json")
files.download("clicksy_dataset.csv")
files.download("eda_plot.png")
files.download("k_selection.png")
files.download("predictions_plot.png")

print("All files downloaded ✅")

## Step 12 — Quick Prediction Test
Test the model with a few sample inputs to verify it works correctly before deploying.

In [ ]:
def predict_price(brand, category, condition, purchase_year, sale_year=2025):
    CONDITION_SCORE_MAP = {
        "New (Open Box)": 6, "Like New": 5, "Excellent": 4,
        "Good": 3, "Fair": 2, "For Parts": 1,
    }
    # Category median new prices (used when exact model unknown)
    CAT_MEDIAN_PRICE = {
        "Camera Body": 134990, "Lens": 74990, "Drone": 109990,
        "Lighting": 34990, "Audio": 24990, "Stabilizer": 34990, "Accessory": 7990,
    }
    features = np.array([[
        CONDITION_SCORE_MAP.get(condition, 3),
        sale_year - purchase_year,
        np.log1p(CAT_MEDIAN_PRICE.get(category, 50000)),
        CATEGORY_LIST.index(category) if category in CATEGORY_LIST else len(CATEGORY_LIST),
        BRAND_TIER.get(brand, 0.85),
    ]])
    raw = final_pipeline.predict(features)[0]
    return int(round(raw / 500) * 500)

# ── Test cases ────────────────────────────────────────────────────────────────
tests = [
    ("Sony",    "Camera Body", "Good",      2021),
    ("Canon",   "Lens",        "Excellent", 2020),
    ("DJI",     "Drone",       "Like New",  2022),
    ("Godox",   "Lighting",    "Good",      2020),
    ("Rode",    "Audio",       "Excellent", 2021),
    ("DJI",     "Stabilizer",  "Good",      2022),
    ("Nikon",   "Accessory",   "Good",      2021),
    ("Leica",   "Camera Body", "Like New",  2022),
]

print(f"{'Brand':<10} {'Category':<14} {'Condition':<14} {'Year':<6} {'Predicted Price':>16}")
print("-" * 65)
for (brand, cat, cond, year) in tests:
    price = predict_price(brand, cat, cond, year)
    print(f"{brand:<10} {cat:<14} {cond:<14} {year:<6} ₹{price:>14,}")

## Step 13 — Deploy to Hugging Face Spaces

### Files to upload:
| File | Purpose |
|------|---------|
| `clicksy_knn_model.pkl` | Trained sklearn pipeline |
| `model_metadata.json` | Feature config & metrics |
| `app.py` | FastAPI server |
| `requirements.txt` | Python dependencies |
| `Dockerfile` | Container config |

### Hugging Face Steps:
1. Go to [huggingface.co/new-space](https://huggingface.co/new-space)
2. Name it `clicksy-price-predictor`
3. Select **Docker** as SDK
4. Upload all 5 files above
5. Space will auto-build and give you a public URL like:
   `https://YOUR-USERNAME-clicksy-price-predictor.hf.space`
6. Paste that URL into your `knnPricePredictor.js` frontend file

### Test your live API:
```bash
curl -X POST https://YOUR-USERNAME-clicksy-price-predictor.hf.space/predict \
  -H "Content-Type: application/json" \
  -d '{"brand":"Sony","category":"Camera Body","condition":"Good","purchase_year":2021}'
```